# Toronto Neighbourhood Stress & Green Space: A Spatial Data Science Case Study

## Problem
Toronto’s 140+ neighbourhoods vary widely in:
- Chronic disease rates  
- Environmental exposures  
- Access to green space  

These datasets are siloed, making it hard to see **where stress is highest** and how environmental factors play a role.  

**Research Question:**  
*Does access to parks influence neighbourhood-level stress, and how does spatial context shape that relationship?*

---

## Data

**Sources:**
- **Mental Health Indicators** — Toronto Public Health (cluster-level stress, coping, satisfaction scores)  
- **Green Space Inventory** — City of Toronto (polygons for all parks)  
- **Neighbourhood Census Profiles (2021)** — Statistics Canada (demographics, socioeconomic data)  

**Challenges solved:**
- Reconciling **inconsistent neighbourhood names**
- Aggregating **different spatial scales** (cluster → neighbourhood)
- Managing **collinearity** in census features

---

## Approach

1. **Data Integration & Cleaning**
   - Merged 3 datasets into a unified **neighbourhood-level GeoDataFrame**.
   - Created **three green-space metrics**:  
     1. Park count  
     2. Park area per person  
     3. Log-distance to nearest park

2. **Feature Engineering**
   - Computed demographic rates (child %, immigrant %, education level).
   - Used **PCA** to combine *avg household size* + *child %* into `hh_child_pc1` to address multicollinearity.

3. **Exploratory Data Analysis**
   - Screened correlations between green-space metrics and stress.
   - Chose `stress_pct` as dependent variable (highest association with green space, r ≈ –0.19).

4. **Modeling**
   - **Baseline:** OLS regression with `park_area_log`, `hh_child_pc1`, `edu_pct`.
   - Tested **Moran’s I** → found strong spatial autocorrelation (I ≈ 0.39).
   - **Advanced:** Spatial-Lag regression (ML_Lag, PySAL) to model neighbourhood spillover effects.

5. **Model Selection**
   - Compared metrics (`park_area_log` vs `park_dist_log`).
   - Used **AIC, R², and 5-fold CV RMSE** for evaluation.
   - Tested splines & interactions — retained simpler model for interpretability.


---

## Results

**Best Model:** Spatial-Lag regression with `log-distance to nearest park` + `hh_child_pc1` + `edu_pct`.

| Metric             | OLS RMSE | Spatial-Lag RMSE | R² Adj (OLS) | Pseudo-R² (Lag) |
|--------------------|----------|------------------|--------------|-----------------|
| park_area_log      | 1.52     | 1.07             | 0.31         | 0.45            |
| park_dist_log      | 1.48     | **1.05**         | 0.33         | **0.46**        |

**Interpretation:**
- Greater distance to parks → higher reported stress (p < 0.05).
- Household structure (`hh_child_pc1`) is the **strongest predictor** — larger, child-heavy households = lower stress.
- Accounting for spatial spillover **nearly doubled explanatory power**.

---

## Impact

- **Urban Planning:** Spatial-lag analysis pinpoints *where* interventions matter most, avoiding “one-size-fits-all” policy.
- **Public Health:** Highlights the role of neighbourhood context in stress reduction. Parks matter, but so does family-friendly infrastructure.
- **Methodological:** Demonstrates rigorous integration of **GIS**, **census**, and **health** data with spatial econometrics.

---

## Limitations

- **Ecological scope:** Results apply at neighbourhood level, not individuals.
- **Missing variables:** Park quality, walkability, noise, social cohesion not included.
- **Collinearity:** Some census features still moderately correlated after PCA.

---

## Next Steps

- Add **park quality indices** or walkability scores.
- Explore **other health outcomes** (e.g., sense of belonging, physical activity rates).
- Extend workflow to **other cities** for comparative analysis.
- Combine with **qualitative research** (resident interviews) for deeper insights.

---

## Technical Stack

- **Data Handling:** pandas, geopandas  
- **Spatial Analysis:** PySAL (`Moran`, `ML_Lag`), shapely  
- **Statistical Modeling:** statsmodels, scikit-learn  
- **Visualization:** matplotlib, GeoPandas plotting  
- **Feature Engineering:** PCA, transformations (log, winsorization)  

---

## Bottom Line

While proximity to parks *does* relate to lower stress, **household composition and spatial context matter far more**.  
Spatial-lag modeling reveals that stress “spreads” between neighbouring areas, making **spatially targeted interventions** essential for improving urban mental health.
